<a href="https://colab.research.google.com/github/sensein/asd-ai-scoping-review/blob/update-scripts/scripts/PRISMA_pipeline_Fabio/2_remove_duplicates.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
import pandas as pd
# Mount Google Drive
from google.colab import drive as gdrive
import string
import re

In [ ]:
gdrive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define the folder containing the subfolders with CSV files
main_folder = '/content/drive/MyDrive/[ICON fellowship] social robot and autism/AI for behavioral analysis and autism/queries'

# Define the target CSV file name
target_csv_name = 'autism_formatted__controlled_combined.csv'

In [ ]:

df = pd.read_csv(f"{main_folder}/{target_csv_name}")

In [ ]:
print(len(df))

162986


In [ ]:
# Replace "[No abstract available]" with None in the "Abstract" column
df['Abstract'] = df['Abstract'].replace('[No abstract available]', None)
#print(len(df))

In [ ]:
# Create a boolean mask for rows that don't start with "NOTICE OF RETRACTION:"
mask = ~df['Abstract'].str.startswith('NOTICE OF RETRACTION:', na=False)

# Apply the mask to filter the DataFrame
df = df[mask]

# Reset the index
df.reset_index(drop=True, inplace=True)
print(len(df))

162986


In [ ]:
df

,Title,Abstract,Keywords,DOI,URL,Authors,Venue,Year,Database
0,Perspectives on children’s autistic traits in ...,Background\nWhile most autism research is cond...,"Autism spectrum disorder, Autism, Autistic tra...",https://doi.org/10.1016/j.ridd.2023.104576,https://www.sciencedirect.com/science/article/...,Elisa Genovesi and Philippa Ullmer and Laila B...,Research in Developmental Disabilities,2023.0,sciencedirect
1,Chronic inhibition of astrocytic aquaporin-4 i...,"Social communication and interaction deficits,...","Autism spectrum disorder, Valproic acid, Aquap...",https://doi.org/10.1016/j.physbeh.2023.114286,https://www.sciencedirect.com/science/article/...,Shima Davoudi and Mona Rahdar and Narges Hosse...,Physiology & Behavior,2023.0,sciencedirect
2,Touch in learning interactions with autistic c...,Learning is fundamentally based on the partici...,"Attention, Engagement, Instruction, Interperso...",https://doi.org/10.1016/j.lcsi.2023.100731,https://www.sciencedirect.com/science/article/...,Vivien Heller,"Learning, Culture and Social Interaction",2023.0,sciencedirect
3,"Association of GABRG3, GABRB3, HTR2A gene vari...",Autism spectrum disorder (ASD) is a neurodevel...,"Autism (ASD), , , , RFLP",https://doi.org/10.1016/j.gene.2023.147399,https://www.sciencedirect.com/science/article/...,Ender M. Coskunpinar and Seymanur Tur and Nagi...,Gene,2023.0,sciencedirect
4,Sialic acid and anti-ganglioside M1 antibodies...,Background\nAutism spectrum disorders (ASD) ar...,"Autism spectrum disorder, Autoimmune, Anti-GM1...",https://doi.org/10.1016/j.braindev.2022.11.006,https://www.sciencedirect.com/science/article/...,Engy A. Ashaat and Sahar Sabry and Moushira E....,Brain and Development,2023.0,sciencedirect
...,...,...,...,...,...,...,...,...,...
162981,Soft Systems Methodology as a Tool to Aid a Pi...,"Guided by soft systems methodology, in an 8-mo...","soft systems, socially assistive robot, soft s...",NaN,NaN,"Lewis, Lundy and Charron, Nancy and Clamp, Chr...",The Eleventh ACM/IEEE International Conference...,2016.0,acm_digital_library
162982,Providing and Accessing Support During the COV...,"Due to the COVID-19 pandemic, essential servic...","telehealth, community centers, essential servi...",10.1145/3441852.3476470,https://doi.org/10.1145/3441852.3476470,"Thang, Tiffany and Liang, Alice and Choi, Yech...",Proceedings of the 23rd International ACM SIGA...,2021.0,acm_digital_library
162983,Engagement: The Inputs and the Outputs: Confer...,The Inputs-Outputs interdisciplinary conferenc...,"cognitive absorption, interest, motivation, im...",10.1145/2557595.2557596,https://doi.org/10.1145/2557595.2557596,"Witchel, Harry J.",Proceedings of the 2013 Inputs-Outputs Confere...,2013.0,acm_digital_library
162984,Hyper Sensorial -- Human Computed Neurodiverge...,"In this video artwork, the author looks at the...","computing, psychology, science, neurodiversity...",10.1145/3290607.3311779,https://doi.org/10.1145/3290607.3311779,"Damiani, Luca M.",Extended Abstracts of the 2019 CHI Conference ...,2019.0,acm_digital_library


In [ ]:
# Remove punctuation
def remove_punctuation(text):
    return f"{text}".translate(str.maketrans('', '', string.punctuation))

# Remove extra spaces
def remove_extra_spaces(text):
    return re.sub(' +', ' ', text)

# Custom function to identify duplicates considering None values
def drop_duplicates_preserve_none(df, column):

    # Remove punctuation and extra spaces from the "Abstract" column
    df['utility'] = df[column].str.translate(str.maketrans('', '', string.punctuation)).str.strip()

    # Create a mask to identify rows where "Abstract" is not None
    not_none_mask = df[column].notna()

    # Apply drop_duplicates only to the rows where "Abstract" is not None
    df_filtered = df[not_none_mask].drop_duplicates(subset=['utility'])

    # Concatenate the rows where "Abstract" is None with the filtered DataFrame
    result_df = pd.concat([df[df[column].isna()], df_filtered], axis=0)

    # Drop the 'B' column in place
    result_df.drop('utility', axis=1, inplace=True)

    return result_df

# Custom function to identify and print duplicates
def print_duplicates_preserve_none(df, column):
    # Create a mask to identify rows where "Abstract" is not None
    not_none_mask = df[column].notna()

    # Filter the DataFrame to rows where "Abstract" is not None
    df_filtered = df[not_none_mask]

    # Identify and print duplicates within the filtered DataFrame
    duplicates = df_filtered[df_filtered.duplicated(subset=[column], keep=False)]
    duplicates = duplicates.sort_values(by=[column])

    #print("Duplicate rows based on the 'Abstract' column:")
    #print(duplicates)

    # Concatenate the rows where "Abstract" is None with the filtered DataFrame
    result_df = pd.concat([df[df[column].isna()], df_filtered], axis=0)

    return duplicates

In [ ]:
# Use the custom function to drop duplicates while preserving None values
print(len(df))
df = drop_duplicates_preserve_none(df, 'DOI').reset_index(drop=True)
print(len(df))
#df = drop_duplicates_preserve_none(df, 'URL').reset_index(drop=True)
#print(len(df))
#df = drop_duplicates_preserve_none(df, 'Abstract').reset_index(drop=True)
#print(len(df))

162986
114879


In [ ]:
# Save the combined DataFrame to the target CSV file
df.to_csv(f'{main_folder}/autism_formatted__controlled_combined_noduplicates.csv', index=False)